# Importar alunos do SigEduca para o Refeitório (SisGI), uma DRE por vez

Grava em `refeitorio_aluno` os alunos de **todas as escolas de uma DRE**, a partir do arquivo do SigEduca.
Usa as mesmas regras da importação do SisGI (`refeitorio_importar_sigeduca.php`):

- `id_aluno` = **AlunoID** do SigEduca (só números, sem zero à esquerda). Chave: escola + id_aluno.
- Nome em maiúsculas; turma = série + turma, como no SisGI.
- Aluno repetido na mesma escola: vale a primeira linha.
- Aluno que já existe: nome e turma são atualizados e ele é reativado.
- **Inativação:** quem está ativo na escola mas não veio no arquivo é inativado (transferido). Proteção: se o arquivo
  trouxer menos da metade dos alunos ativos de hoje da escola, ninguém dessa escola é inativado (exportação incompleta).
- Escola que não está no arquivo não é alterada.

**Como usar:** escolha a DRE na célula 1, rode tudo com `SIMULAR = True`, confira o plano da célula 5 e, se estiver certo,
mude para `SIMULAR = False` e rode de novo. Tudo é gravado numa única transação (ou tudo, ou nada) e fica registrado na
auditoria do SisGI.

**LGPD:** só são lidas as colunas usadas (escola, AlunoID, nome, série e turma). As saídas mostram apenas totais por escola.
**Senhas:** nenhuma senha fica no notebook. Com login SQL, ela é pedida na hora (`getpass`).

## 1. Configuração

In [125]:
# Arquivo do SigEduca (o geral ou o de uma DRE gerado por separar_por_dre.ipynb)
ARQUIVO = 'relatorio_matriculas_2026.csv'

# DREs do arquivo do SigEduca (nomes exatamente como aparecem na coluna DRE)
DRES = {
    1:  'DRE ALTA FLORESTA',
    2:  'DRE BARRA DO GARCAS',
    3:  'DRE CACERES',
    4:  'DRE CONFRESA',
    5:  'DRE DIAMANTINO',
    6:  'DRE JUINA',
    7:  'DRE MATUPA',
    8:  'DRE METROPOLITANA',
    9:  'DRE PONTES E LACERDA',
    10: 'DRE PRIMAVERA DO LESTE',
    11: 'DRE RONDONOPOLIS',
    12: 'DRE SINOP',
    13: 'DRE TANGARA DA SERRA',
}

# DRE a importar: troque o número (1 a 13)
DRE = DRES[12]

SIMULAR = False             # True = só mostra o plano; False = grava no banco
INATIVAR_AUSENTES = True   # inativa quem saiu da escola (não está no arquivo)
FORCAR = False             # inativa mesmo quando o arquivo tem menos da metade dos ativos de hoje

# Banco de dados
SERVIDOR = 'localhost'     # trabalho: 'descartes.seduc.mt.gov.br\\mssql_des02' (use o nome, não o IP)
BANCO = 'sisgi'
LOGIN_SQL = ''             # vazio = autenticação do Windows; preenchido = login SQL (a senha é pedida na hora)
CONFIAR_CERTIFICADO = SERVIDOR.lower() in ('localhost', '.', '127.0.0.1')  # certificado próprio do SQL de casa

## 2. Leitura do arquivo e DREs disponíveis

In [126]:
import re, json, getpass, socket, warnings
import pandas as pd
pd.set_option('display.max_rows', 10)
pd.set_option('display.precision', 2)
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')  # pd.read_sql com pyodbc funciona normalmente

COLUNAS = ['DRE', 'Municipio', 'LotacaoID', 'Lotacao', 'AlunoID', 'Nome', 'RelacaoSerie', 'Turma']
dados = pd.read_csv(ARQUIVO, sep=';', dtype=str, keep_default_na=False, usecols=COLUNAS, encoding='utf-8-sig')
print(f'{len(dados):,} linhas no arquivo'.replace(',', '.'))
# Confere o dicionário DRES com as DREs do arquivo
fora_do_dicionario = sorted(set(dados['DRE']) - set(DRES.values()))
fora_do_arquivo = sorted(set(DRES.values()) - set(dados['DRE']))
if fora_do_dicionario: print('ATENÇÃO: DRE(s) no arquivo que não estão em DRES:', fora_do_dicionario)
if fora_do_arquivo: print('ATENÇÃO: DRE(s) de DRES que não estão no arquivo:', fora_do_arquivo)
print(f'DRE escolhida: {DRE}')
numero = {v: k for k, v in DRES.items()}
(dados.assign(Numero=dados['DRE'].map(numero)).groupby(['Numero', 'DRE'], dropna=False).agg(Escolas=('LotacaoID', 'nunique'), Linhas=('AlunoID', 'size'))
      .sort_index())

313.647 linhas no arquivo
DRE escolhida: DRE SINOP


,,Escolas,Linhas
Numero,DRE,,
1,DRE ALTA FLORESTA,26,10195
2,DRE BARRA DO GARCAS,58,20603
3,DRE CACERES,37,15532
4,DRE CONFRESA,35,11558
5,DRE DIAMANTINO,43,15273
...,...,...,...
9,DRE PONTES E LACERDA,22,10854
10,DRE PRIMAVERA DO LESTE,40,18642
11,DRE RONDONOPOLIS,60,32659


## 3. Alunos da DRE escolhida (mesmas regras do SisGI)

In [127]:
assert DRE, 'Escolha a DRE na célula 1 (variável DRE), com o nome exatamente como aparece na lista acima.'
assert DRE in set(dados['DRE']), f'A DRE {DRE!r} não está no arquivo. Confira o nome na lista acima.'

def espacos(v):
    return ' '.join(str(v).split())

def id_aluno(v):
    v = str(v).strip()
    return int(v) if re.fullmatch(r'[1-9][0-9]{0,9}', v) and int(v) <= 2147483647 else None

def turma_sisgi(serie, turma):
    # Igual a refeitorioSigEducaTurma() do SisGI
    serie, turma = espacos(serie), espacos(turma)
    if turma == '':
        return serie[:50].upper() or None
    m = re.match(r'\d+', serie)
    if serie == '' or (m and re.search(r'(^|\D)' + m.group(0) + r'\D', turma + ' ')):
        return turma[:50].upper()
    return (serie + ' - ' + turma)[:50].upper()

d = dados[dados['DRE'] == DRE].copy()
d['id_lotacao'] = pd.to_numeric(d['LotacaoID'], errors='coerce')
d['id_aluno'] = d['AlunoID'].map(id_aluno)
d['nome_aluno'] = d['Nome'].map(espacos).str.upper()
d['turma'] = [turma_sisgi(s, t) for s, t in zip(d['RelacaoSerie'], d['Turma'])]

erros = {
    'LotacaoID inválido': d['id_lotacao'].isna(),
    'AlunoID inválido': d['id_aluno'].isna(),
    'nome vazio ou com mais de 150 letras': (d['nome_aluno'] == '') | (d['nome_aluno'].str.len() > 150),
}
invalida = pd.Series(False, index=d.index)
for motivo, m in erros.items():
    if m.sum(): print(f'{m.sum()} linha(s) ignorada(s): {motivo}')
    invalida |= m
d = d[~invalida]
antes = len(d)
d = d.drop_duplicates(subset=['id_lotacao', 'id_aluno'], keep='first')
print(f'{antes - len(d)} linha(s) repetida(s) do mesmo aluno na mesma escola descartada(s)')

carga = d[['id_lotacao', 'id_aluno', 'nome_aluno', 'turma']].astype({'id_lotacao': 'int64', 'id_aluno': 'int64'})
escolas_arquivo = (d.groupby(['id_lotacao', 'Lotacao', 'Municipio']).size().rename('no_arquivo').reset_index()
                    .astype({'id_lotacao': 'int64'}))
print(f'{DRE}: {len(carga):,} alunos em {len(escolas_arquivo)} escolas'.replace(',', '.'))

0 linha(s) repetida(s) do mesmo aluno na mesma escola descartada(s)
DRE SINOP: 43.671 alunos em 58 escolas


## 4. Conexão com o banco

In [128]:
import pyodbc

driver = next((x for x in ['ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server'] if x in pyodbc.drivers()), None)
assert driver, 'Instale o ODBC Driver 17 ou 18 for SQL Server.'
partes = [f'DRIVER={{{driver}}}', f'SERVER={SERVIDOR}', f'DATABASE={BANCO}', 'Encrypt=yes',
          f'TrustServerCertificate={"yes" if CONFIAR_CERTIFICADO else "no"}', 'APP=SisGI importacao por DRE']
if LOGIN_SQL:
    partes += [f'UID={LOGIN_SQL}', 'PWD=' + getpass.getpass(f'Senha de {LOGIN_SQL}: ')]
else:
    partes.append('Trusted_Connection=yes')
conn = pyodbc.connect(';'.join(partes), autocommit=False)
partes = None  # não mantém a senha em memória
cur = conn.cursor()

# A tabela precisa estar na estrutura nova (id_aluno = AlunoID): script refeitorio_id_aluno_sigeduca.sql
cur.execute("SELECT COL_LENGTH('dbo.refeitorio_aluno', 'id_aluno'), COL_LENGTH('dbo.refeitorio_aluno', 'matricula')")
tem_id, tem_matricula = cur.fetchone()
assert tem_id and not tem_matricula, 'Rode antes o script_atualiza/refeitorio_id_aluno_sigeduca.sql neste banco.'
cur.execute('SELECT @@SERVERNAME, DB_NAME(), SUSER_SNAME()')
print('Conectado: servidor %s, banco %s, login %s' % tuple(cur.fetchone()))

Conectado: servidor Principal, banco sisgi, login PRINCIPAL\jhony


## 5. Plano da importação (nada é gravado aqui)

Os alunos vão para uma tabela temporária e o plano é calculado no banco, escola por escola.

In [129]:
cur.execute('''IF OBJECT_ID('tempdb..#carga') IS NOT NULL DROP TABLE #carga;
               CREATE TABLE #carga (id_lotacao INT NOT NULL, id_aluno INT NOT NULL,
                                    nome_aluno NVARCHAR(150) COLLATE DATABASE_DEFAULT NOT NULL,
                                    turma NVARCHAR(50) COLLATE DATABASE_DEFAULT NULL,
                                    PRIMARY KEY (id_lotacao, id_aluno));''')
# Cursor só para a carga em massa (os tipos fixados aqui não valem para os outros comandos)
cur_carga = conn.cursor()
cur_carga.fast_executemany = True
cur_carga.setinputsizes([(pyodbc.SQL_INTEGER, 0, 0), (pyodbc.SQL_INTEGER, 0, 0), (pyodbc.SQL_WVARCHAR, 150, 0), (pyodbc.SQL_WVARCHAR, 50, 0)])
linhas = [(int(a), int(b), n, t) for a, b, n, t in carga.itertuples(index=False, name=None)]
for i in range(0, len(linhas), 5000):
    cur_carga.executemany('INSERT INTO #carga (id_lotacao, id_aluno, nome_aluno, turma) VALUES (?, ?, ?, ?)', linhas[i:i + 5000])
cur_carga.close()
linhas = None

plano = pd.read_sql('''
    SELECT e.id_lotacao,
           CASE WHEN l.id_lotacao IS NULL THEN 0 ELSE 1 END AS existe_no_sisgi,
           (SELECT COUNT(*) FROM #carga c WHERE c.id_lotacao = e.id_lotacao) AS no_arquivo,
           (SELECT COUNT(*) FROM dbo.refeitorio_aluno a WHERE a.id_lotacao = e.id_lotacao AND a.ativo = 1) AS ativos_hoje,
           (SELECT COUNT(*) FROM #carga c WHERE c.id_lotacao = e.id_lotacao
              AND NOT EXISTS (SELECT 1 FROM dbo.refeitorio_aluno a WHERE a.id_lotacao = c.id_lotacao AND a.id_aluno = c.id_aluno)) AS novos,
           (SELECT COUNT(*) FROM #carga c INNER JOIN dbo.refeitorio_aluno a ON a.id_lotacao = c.id_lotacao AND a.id_aluno = c.id_aluno
             WHERE c.id_lotacao = e.id_lotacao) AS atualizados,
           (SELECT COUNT(*) FROM #carga c INNER JOIN dbo.refeitorio_aluno a ON a.id_lotacao = c.id_lotacao AND a.id_aluno = c.id_aluno
             WHERE c.id_lotacao = e.id_lotacao AND a.ativo = 0) AS reativados,
           (SELECT COUNT(*) FROM dbo.refeitorio_aluno a WHERE a.id_lotacao = e.id_lotacao AND a.ativo = 1
              AND NOT EXISTS (SELECT 1 FROM #carga c WHERE c.id_lotacao = a.id_lotacao AND c.id_aluno = a.id_aluno)) AS ausentes
    FROM (SELECT DISTINCT id_lotacao FROM #carga) e
    LEFT JOIN dbo.lotacao l ON l.id_lotacao = e.id_lotacao''', conn)

plano = escolas_arquivo[['id_lotacao', 'Lotacao', 'Municipio']].merge(plano, on='id_lotacao')
# Proteção contra exportação incompleta (mesma regra do importador do SisGI)
incompleto = (plano['ativos_hoje'] > 0) & (plano['no_arquivo'] < plano['ativos_hoje'] / 2)
plano['inativar'] = INATIVAR_AUSENTES & (plano['existe_no_sisgi'] == 1) & (~incompleto | FORCAR)
plano['a_inativar'] = plano['ausentes'].where(plano['inativar'], 0)
plano['aviso'] = ''
plano.loc[plano['existe_no_sisgi'] == 0, 'aviso'] = 'escola não cadastrada no SisGI: ignorada'
plano.loc[(plano['existe_no_sisgi'] == 1) & incompleto & INATIVAR_AUSENTES & (not FORCAR),
          'aviso'] = 'arquivo com menos da metade dos ativos: ninguém inativado'

validas = plano[plano['existe_no_sisgi'] == 1]
print(f'{DRE}: {len(validas)} escola(s) | novos {validas.novos.sum()} | atualizados {validas.atualizados.sum()} '
      f'(reativados {validas.reativados.sum()}) | a inativar {validas.a_inativar.sum()}')
if (plano['existe_no_sisgi'] == 0).any():
    print(f'{(plano.existe_no_sisgi == 0).sum()} escola(s) do arquivo não existem na tabela lotacao e serão ignoradas.')
pd.set_option('display.max_rows', 300)
plano.drop(columns=['inativar']).sort_values(['aviso', 'Municipio', 'Lotacao'], ascending=[False, True, True]).reset_index(drop=True)

DRE SINOP: 58 escola(s) | novos 43671 | atualizados 0 (reativados 0) | a inativar 0


,id_lotacao,Lotacao,Municipio,existe_no_sisgi,no_arquivo,ativos_hoje,novos,atualizados,reativados,ausentes,a_inativar,aviso
0,124028,EEI EDUCAÇÃO BASICA MAYROWI APIAKA,APIACAS ...,1,69,0,69,0,0,0,0,
1,116033,EE CRISTIANO ARAÚJO PIRES,BOA ESPERANCA DO NORTE ...,1,518,0,518,0,0,0,0,
2,172456,EE DOROTHY STANG,CLAUDIA ...,1,104,0,104,0,0,0,0,
3,172081,EE FLORESTAN FERNANDES,CLAUDIA ...,1,58,0,58,0,0,0,0,
4,13374,EE MANOEL SOARES CAMPOS,CLAUDIA ...,1,848,0,848,0,0,0,0,
5,158330,EE ANDRÉ ANTONIO MAGGI,COLIDER ...,1,486,0,486,0,0,0,0,
6,13471,EE CAFENORTE,COLIDER ...,1,79,0,79,0,0,0,0,
7,13404,EE DESEMBARGADOR MILTON ARMANDO POMPEU DE BARROS,COLIDER ...,1,705,0,705,0,0,0,0,
8,13412,EE MILITAR DOM PEDRO II 2º TEN BM KLEIBER RODR...,COLIDER ...,1,445,0,445,0,0,0,0,
9,13420,EE NOVA GALILEIA,COLIDER ...,1,51,0,51,0,0,0,0,


## 6. Gravação

Só grava com `SIMULAR = False`. Uma transação para a DRE inteira: se der erro, nada fica gravado.

In [130]:
if SIMULAR:
    print('SIMULAÇÃO: nada foi gravado. Confira o plano acima e rode de novo com SIMULAR = False.')
else:
    ids_inativar = [int(x) for x in plano.loc[plano['inativar'] & (plano['ausentes'] > 0), 'id_lotacao']]
    try:
        # Só escolas que existem no SisGI
        cur.execute('DELETE c FROM #carga c WHERE NOT EXISTS (SELECT 1 FROM dbo.lotacao l WHERE l.id_lotacao = c.id_lotacao)')
        cur.execute('''UPDATE a SET nome_aluno = c.nome_aluno, turma = c.turma, ativo = 1, origem = 'PLANILHA',
                                    data_atualizacao = GETDATE(), id_usuario_atualizacao = NULL
                       FROM dbo.refeitorio_aluno a INNER JOIN #carga c ON c.id_lotacao = a.id_lotacao AND c.id_aluno = a.id_aluno''')
        atualizados = cur.rowcount
        cur.execute('''INSERT INTO dbo.refeitorio_aluno (id_lotacao, id_aluno, nome_aluno, turma, origem, id_usuario_atualizacao)
                       SELECT c.id_lotacao, c.id_aluno, c.nome_aluno, c.turma, 'PLANILHA', NULL
                       FROM #carga c
                       WHERE NOT EXISTS (SELECT 1 FROM dbo.refeitorio_aluno a WHERE a.id_lotacao = c.id_lotacao AND a.id_aluno = c.id_aluno)''')
        novos = cur.rowcount
        inativados = 0
        if ids_inativar:
            cur.execute('IF OBJECT_ID(\'tempdb..#inativar\') IS NOT NULL DROP TABLE #inativar; CREATE TABLE #inativar (id_lotacao INT PRIMARY KEY);')
            cur.executemany('INSERT INTO #inativar (id_lotacao) VALUES (?)', [(x,) for x in ids_inativar])
            cur.execute('''UPDATE a SET ativo = 0, data_atualizacao = GETDATE(), id_usuario_atualizacao = NULL
                           FROM dbo.refeitorio_aluno a INNER JOIN #inativar i ON i.id_lotacao = a.id_lotacao
                           WHERE a.ativo = 1
                             AND NOT EXISTS (SELECT 1 FROM #carga c WHERE c.id_lotacao = a.id_lotacao AND c.id_aluno = a.id_aluno)''')
            inativados = cur.rowcount
        # Auditoria do SisGI (sem dados de alunos: só totais por escola)
        resumo = {'arquivo': ARQUIVO.replace('\\', '/').split('/')[-1], 'dre': DRE, 'novos': novos, 'atualizados': atualizados,
                  'inativados': inativados, 'escolas': {str(r.id_lotacao): {'no_arquivo': int(r.no_arquivo), 'inativar': bool(r.inativar)}
                                                        for r in plano[plano.existe_no_sisgi == 1].itertuples()}}
        cur.execute('''INSERT INTO dbo.seguranca_auditoria (id_usuario_ator, nome_usuario_ator, acao, dados_depois, ip_origem, pagina)
                       VALUES (NULL, ?, 'REFEITORIO_IMPORTACAO_SIGEDUCA', ?, NULL, 'importar_alunos_refeitorio.ipynb')''',
                    f'Jupyter: {getpass.getuser()}@{socket.gethostname()}'[:150], json.dumps(resumo, ensure_ascii=False))
        conn.commit()
        print(f'{DRE}: gravado. {novos} novo(s), {atualizados} atualizado(s), {inativados} inativado(s).')
    except Exception:
        conn.rollback()
        print('ERRO: nada foi gravado (transação desfeita).')
        raise

DRE SINOP: gravado. 43671 novo(s), 0 atualizado(s), 0 inativado(s).


## 7. Situação no banco depois da importação

In [131]:
ids = ','.join(str(int(x)) for x in escolas_arquivo['id_lotacao'])   # números inteiros: seguro no SQL
situacao = pd.read_sql(f'''SELECT id_lotacao, SUM(CASE WHEN ativo = 1 THEN 1 ELSE 0 END) AS ativos,
                                  SUM(CASE WHEN ativo = 0 THEN 1 ELSE 0 END) AS inativos
                           FROM dbo.refeitorio_aluno WHERE id_lotacao IN ({ids}) GROUP BY id_lotacao''', conn)
situacao = escolas_arquivo.merge(situacao, on='id_lotacao', how='left').fillna({'ativos': 0, 'inativos': 0})
print(f'{DRE}: {int(situacao.ativos.sum())} aluno(s) ativo(s) e {int(situacao.inativos.sum())} inativo(s) no SisGI')
situacao.sort_values(['Municipio', 'Lotacao']).reset_index(drop=True)

DRE SINOP: 43671 aluno(s) ativo(s) e 0 inativo(s) no SisGI


,id_lotacao,Lotacao,Municipio,no_arquivo,ativos,inativos
0,124028,EEI EDUCAÇÃO BASICA MAYROWI APIAKA,APIACAS ...,69,69,0
1,116033,EE CRISTIANO ARAÚJO PIRES,BOA ESPERANCA DO NORTE ...,518,518,0
2,172456,EE DOROTHY STANG,CLAUDIA ...,104,104,0
3,172081,EE FLORESTAN FERNANDES,CLAUDIA ...,58,58,0
4,13374,EE MANOEL SOARES CAMPOS,CLAUDIA ...,848,848,0
5,158330,EE ANDRÉ ANTONIO MAGGI,COLIDER ...,486,486,0
6,13471,EE CAFENORTE,COLIDER ...,79,79,0
7,13404,EE DESEMBARGADOR MILTON ARMANDO POMPEU DE BARROS,COLIDER ...,705,705,0
8,13412,EE MILITAR DOM PEDRO II 2º TEN BM KLEIBER RODR...,COLIDER ...,445,445,0
9,13420,EE NOVA GALILEIA,COLIDER ...,51,51,0


## 8. Encerrar

Fecha a conexão. Para importar outra DRE, troque `DRE` na célula 1 e rode tudo de novo (com `SIMULAR = True` primeiro).

In [132]:
cur.close()
conn.close()
print('Conexão fechada.')

Conexão fechada.
